# Financial Data Analysis — Medal Days × Markets (ALL / SUMMER / WINTER)

This notebook takes the **three medal-day CSVs** produced in Notebook 1:

1. `usa_medal_daily_counts_and_score_ALL.csv`
2. `usa_medal_daily_counts_and_score_SUMMER.csv`
3. `usa_medal_daily_counts_and_score_WINTER.csv`

…and for each dataset:

- maps medal dates to the **next NYSE trading day**
- downloads market data for selected tickers (default: `SPY`, `NKE`, `XLY`, `DKNG`)
- computes daily **log returns**, **log volume change**, and **volatility proxies**
- computes **abnormal returns** (rolling **CAPM** and rolling **FF3**) for tickers excluding `SPY`
- merges everything into one regression-ready panel
- saves outputs per dataset: `ALL`, `SUMMER`, `WINTER`

Outputs (created when you run the notebook):
- `usa_medals_market_panel_ALL.csv`, `usa_medals_market_panel_SUMMER.csv`, `usa_medals_market_panel_WINTER.csv`
- `usa_medals_market_panel_regression_subset_ALL.csv`, etc.

> If you only want to run one dataset (e.g., SUMMER), set `RUN_WHICH = ["SUMMER"]` below.


In [ ]:
# Install dependencies if needed (comment out if already installed)
%pip -q install pandas numpy yfinance pandas_datareader pandas_market_calendars statsmodels tqdm

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_market_calendars as mcal
from pandas_datareader import data as pdr
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS

pd.set_option("display.max_columns", 250)

## 1) Configuration

- Choose which medal datasets to process (`ALL`, `SUMMER`, `WINTER`).
- Choose tickers.
- Choose rolling window length for abnormal return estimation.


In [ ]:
# Which datasets to run
RUN_WHICH = ["ALL", "SUMMER", "WINTER"]  # change to e.g. ["SUMMER"]

# Medal CSV file names
MEDAL_FILES = {
    "ALL":    ["usa_medal_daily_counts_and_score_ALL.csv",    "/mnt/data/usa_medal_daily_counts_and_score_ALL.csv"],
    "SUMMER": ["usa_medal_daily_counts_and_score_SUMMER.csv", "/mnt/data/usa_medal_daily_counts_and_score_SUMMER.csv"],
    "WINTER": ["usa_medal_daily_counts_and_score_WINTER.csv", "/mnt/data/usa_medal_daily_counts_and_score_WINTER.csv"],
}

# Market tickers
TICKERS = ["SPY", "NKE", "XLY", "DKNG"]  # edit as desired

# Rolling estimation window for CAPM/FF3 abnormal returns
ROLLING_WINDOW = 252  # ~1 year; try 120 if your sample is shorter

# NYSE calendar
nyse = mcal.get_calendar("NYSE")

## 2) Helper functions

In [ ]:
def find_existing_path(candidates):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    return None

def load_medal_panel(which: str) -> pd.DataFrame:
    path = find_existing_path(MEDAL_FILES[which])
    if path is None:
        raise FileNotFoundError(f"Could not find medal file for {which}: {MEDAL_FILES[which]}")
    df = pd.read_csv(path)
    # Normalize date column
    if "event_date_end" in df.columns:
        df = df.rename(columns={"event_date_end": "medal_date"})
    elif "date" in df.columns:
        df = df.rename(columns={"date": "medal_date"})
    elif "medal_date" not in df.columns:
        raise ValueError(f"{which}: expected a date column 'event_date_end' or 'medal_date' or 'date'")
    df["medal_date"] = pd.to_datetime(df["medal_date"], errors="coerce")
    df = df.dropna(subset=["medal_date"]).sort_values("medal_date").reset_index(drop=True)

    # Ensure medal columns exist
    for c in ["gold","silver","bronze"]:
        if c not in df.columns:
            raise ValueError(f"{which}: missing column '{c}'")

    if "total_medals" not in df.columns:
        df["total_medals"] = df[["gold","silver","bronze"]].sum(axis=1)

    if "performance_score" not in df.columns:
        df["performance_score"] = 3*df["gold"] + 2*df["silver"] + 1*df["bronze"]

    # Ensure season column exists if present; otherwise infer from which
    if "season" not in df.columns:
        if which == "SUMMER":
            df["season"] = "S"
        elif which == "WINTER":
            df["season"] = "W"
        else:
            df["season"] = np.nan  # ALL may have mixed; ideally it already exists

    df["any_medal_dummy"] = (df["total_medals"] > 0).astype(int)
    return df

def map_to_next_trading_day(medal_dates: pd.Series) -> pd.Series:
    start = medal_dates.min() - pd.Timedelta(days=10)
    end = medal_dates.max() + pd.Timedelta(days=10)
    schedule = nyse.schedule(start_date=start, end_date=end)
    trading_days = pd.DatetimeIndex(schedule.index)

    def to_next(d: pd.Timestamp):
        if d in trading_days:
            return d
        idx = trading_days.searchsorted(d)
        return trading_days[idx] if idx < len(trading_days) else pd.NaT

    return medal_dates.apply(to_next)

def yfinance_fetch(tickers, start_date, end_date):
    raw = yf.download(tickers, start=str(start_date), end=str(end_date), auto_adjust=False, progress=False)
    prices = raw["Adj Close"] if "Adj Close" in raw else raw["Close"]
    volume = raw["Volume"]

    if isinstance(prices, pd.Series): prices = prices.to_frame()
    if isinstance(volume, pd.Series): volume = volume.to_frame()

    ret = np.log(prices).diff()
    log_vol_change = np.log(volume.replace(0, np.nan)).diff()
    abs_ret = ret.abs()
    vol_20d = ret.rolling(20).std()
    return ret, log_vol_change, abs_ret, vol_20d

def to_long(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    out = df.stack().rename(value_name).reset_index()
    out.columns = ["date", "ticker", value_name]
    return out

def load_ff_factors():
    ff = pdr.DataReader("F-F_Research_Data_Factors_daily", "famafrench")[0].copy()
    ff.index = pd.to_datetime(ff.index)
    ff = ff.rename(columns={"Mkt-RF":"MKT_RF"})
    ff = ff[["MKT_RF","SMB","HML","RF"]].sort_index() / 100.0
    return ff

def compute_abrets_for_ticker(ticker: str, ret_df: pd.DataFrame, ff: pd.DataFrame, window: int) -> pd.DataFrame:
    r = ret_df[ticker].dropna()
    dfm = pd.DataFrame({"ret": r}).join(ff, how="inner").dropna()
    dfm["excess_ret"] = dfm["ret"] - dfm["RF"]
    y = dfm["excess_ret"]

    # CAPM
    Xc = sm.add_constant(dfm[["MKT_RF"]])
    capm = RollingOLS(y, Xc, window=window, min_nobs=max(60, int(window*0.5))).fit()
    capm_pred = (Xc * capm.params).sum(axis=1)
    dfm["abret_capm"] = y - capm_pred

    # FF3
    Xf = sm.add_constant(dfm[["MKT_RF","SMB","HML"]])
    ff3 = RollingOLS(y, Xf, window=window, min_nobs=max(60, int(window*0.5))).fit()
    ff3_pred = (Xf * ff3.params).sum(axis=1)
    dfm["abret_ff3"] = y - ff3_pred

    out = dfm[["abret_capm","abret_ff3"]].copy()
    out["date"] = out.index
    out["ticker"] = ticker
    return out.reset_index(drop=True)


## 3) Load factors once

In [ ]:
ff = load_ff_factors()
display(ff.head())
print("FF factor coverage:", ff.index.min().date(), "→", ff.index.max().date())

## 4) Build panels for ALL / SUMMER / WINTER

In [ ]:
results = {}  # store final panels in-memory

for which in RUN_WHICH:
    print("\n" + "="*80)
    print("PROCESSING:", which)
    medals = load_medal_panel(which)
    print("Loaded medals:", medals.shape)
    display(medals.head())

    # Map to trading days
    medals["trade_date"] = map_to_next_trading_day(medals["medal_date"])
    medals = medals.dropna(subset=["trade_date"]).sort_values("trade_date").reset_index(drop=True)
    trade_dates = pd.DatetimeIndex(sorted(medals["trade_date"].unique()))
    print("Unique trading days:", len(trade_dates))

    # Fetch market data window (1y before first trade date for rolling estimates)
    fetch_start = (trade_dates.min() - pd.Timedelta(days=365)).date()
    fetch_end = (trade_dates.max() + pd.Timedelta(days=10)).date()
    print("Fetching Yahoo Finance:", fetch_start, "→", fetch_end, "Tickers:", TICKERS)

    ret, lvc, abs_ret, vol20 = yfinance_fetch(TICKERS, fetch_start, fetch_end)

    # Build long market table
    market_long = (
        to_long(ret, "ret")
        .merge(to_long(lvc, "log_volume_change"), on=["date","ticker"], how="left")
        .merge(to_long(abs_ret, "abs_ret"), on=["date","ticker"], how="left")
        .merge(to_long(vol20, "vol_20d"), on=["date","ticker"], how="left")
    )

    # Abnormal returns (exclude SPY)
    ab_list = []
    for t in [x for x in TICKERS if x != "SPY"]:
        ab_list.append(compute_abrets_for_ticker(t, ret, ff, window=ROLLING_WINDOW))
    abrets_long = pd.concat(ab_list, ignore_index=True) if ab_list else pd.DataFrame(columns=["date","ticker","abret_capm","abret_ff3"])

    market_long = market_long.merge(abrets_long, on=["date","ticker"], how="left")

    # Restrict to medal trading days and pivot wide
    mkt_td = market_long[market_long["date"].isin(trade_dates)].copy()
    value_cols = ["ret","log_volume_change","abs_ret","vol_20d","abret_capm","abret_ff3"]
    wide = mkt_td.pivot(index="date", columns="ticker", values=value_cols)
    wide.columns = [f"{v}_{t}" for v, t in wide.columns]
    wide = wide.reset_index().rename(columns={"date":"trade_date"})

    final = medals.merge(wide, on="trade_date", how="left")

    # Order core columns
    front = ["trade_date","medal_date","season","gold","silver","bronze","total_medals","performance_score","any_medal_dummy"]
    front = [c for c in front if c in final.columns]
    final = final[front + [c for c in final.columns if c not in front]]

    # Save
    out_panel = Path(f"usa_medals_market_panel_{which}.csv")
    final.to_csv(out_panel, index=False)
    print("Wrote:", out_panel.resolve())

    # Compact regression subset
    keep = [
        "trade_date","medal_date","season","gold","silver","bronze","performance_score","any_medal_dummy",
        "ret_SPY","log_volume_change_SPY","abs_ret_SPY","vol_20d_SPY",
    ]
    for t in [x for x in TICKERS if x != "SPY"]:
        keep += [f"ret_{t}", f"abret_capm_{t}", f"abret_ff3_{t}"]
    keep = [c for c in keep if c in final.columns]

    reg_df = final[keep].copy()
    out_reg = Path(f"usa_medals_market_panel_regression_subset_{which}.csv")
    reg_df.to_csv(out_reg, index=False)
    print("Wrote:", out_reg.resolve())

    results[which] = final

print("\nDone. Panels in memory:", list(results.keys()))